# 4.6 Obstacle Investigation

Interactive map of all extracted clusters.  
**Click any dot** to load its top-view and side-view in the right panels.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from config import CLUSTERS_DIR

In [ ]:
inv = pd.read_csv(CLUSTERS_DIR / 'inventory.csv')
print(f"Loaded {len(inv)} clusters")
print(inv.groupby('label')['cluster_idx'].count().rename('count').to_string())

In [ ]:
LABEL_NAMES = {
    0:  'Unknown',
    30: 'Tree',
    40: 'Car',
    60: 'Street Light',
    83: 'Large Container',
}
LABEL_COLORS = {
    0:  '#888888',
    30: '#44bb44',
    40: '#ff8800',
    60: '#4499ff',
    83: '#cc44cc',
}
DEFAULT_COLOR = '#ffffff'


def hag_colors(npz):
    """Height-above-ground colormap for point scatter."""
    xyz = npz['xyz_centered']
    hag = npz.get('height_ag', None)
    if hag is None or np.all(np.isnan(hag)):
        z = xyz[:, 2]
        hag = z - z.min()
    hag = np.nan_to_num(hag, nan=0.0)
    return plt.cm.plasma(np.clip(hag, 0, 6) / 6.0)


def style_ax(ax):
    ax.set_facecolor('#111111')
    for sp in ax.spines.values():
        sp.set_color('#444444')
    ax.tick_params(colors='#888888', labelsize=7)

In [ ]:
fig = plt.figure(figsize=(17, 7))
fig.patch.set_facecolor('#111111')

gs = gridspec.GridSpec(2, 2, figure=fig,
                       width_ratios=[1.6, 1],
                       hspace=0.45, wspace=0.3)

ax_map  = fig.add_subplot(gs[:, 0])   # left: spatial overview
ax_top  = fig.add_subplot(gs[0, 1])   # top-right: top view
ax_side = fig.add_subplot(gs[1, 1])   # bottom-right: side view

for ax in (ax_map, ax_top, ax_side):
    style_ax(ax)

# ── draw all cluster centroids ────────────────────────────────────────────────
for lbl, grp in inv.groupby('label'):
    color = LABEL_COLORS.get(lbl, DEFAULT_COLOR)
    name  = LABEL_NAMES.get(lbl, f'Label {lbl}')
    ax_map.scatter(
        grp['centroid_x'], grp['centroid_y'],
        c=color, s=70, zorder=3, label=name,
        edgecolors='#222222', linewidths=0.5, alpha=0.9,
    )

legend = ax_map.legend(
    facecolor='#1e1e1e', labelcolor='white',
    edgecolor='#444444', fontsize=8,
)
ax_map.set_title('All clusters — click to inspect', color='white', fontsize=10)
ax_map.set_aspect('equal')
ax_map.set_xlabel('X (m RD)', color='#888888', fontsize=8)
ax_map.set_ylabel('Y (m RD)', color='#888888', fontsize=8)

# selection highlight
sel_marker, = ax_map.plot([], [], 'w*', markersize=16, zorder=5,
                          markeredgecolor='#ff4444', markeredgewidth=1)

# info strip below map
info_text = ax_map.text(
    0.02, 0.02, 'Click a cluster to inspect',
    transform=ax_map.transAxes, color='#aaaaaa',
    fontsize=8, va='bottom',
    bbox=dict(facecolor='#1e1e1e', edgecolor='none', alpha=0.7, pad=3),
)

# placeholder labels
ax_top.set_title('top view', color='#555555', fontsize=9)
ax_side.set_title('side view', color='#555555', fontsize=9)


# ── click handler ─────────────────────────────────────────────────────────────
def on_click(event):
    if event.inaxes is not ax_map:
        return
    if event.xdata is None or event.ydata is None:
        return

    # nearest cluster centroid
    dist = np.hypot(inv['centroid_x'] - event.xdata,
                    inv['centroid_y'] - event.ydata)
    row = inv.loc[dist.idxmin()]

    # move selection marker
    sel_marker.set_data([row['centroid_x']], [row['centroid_y']])

    lbl  = int(row['label'])
    name = LABEL_NAMES.get(lbl, f'Label {lbl}')
    src  = row.get('label_source', '')
    info_text.set_text(
        f"{row['tilecode']}  #cluster {row['cluster_idx']}\n"
        f"{name} ({lbl})  ·  {src}  ·  "
        f"{int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²"
    )

    # load NPZ
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        info_text.set_text(f"Error loading NPZ: {e}")
        fig.canvas.draw_idle()
        return

    xyz  = npz['xyz_centered']
    cols = hag_colors(npz)
    title = (
        f"{row['tilecode']} · #{row['cluster_idx']}  "
        f"{name}  ·  {int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²"
    )

    # top view (XY)
    ax_top.cla()
    style_ax(ax_top)
    ax_top.scatter(xyz[:, 0], xyz[:, 1], c=cols, s=1, linewidths=0)
    ax_top.set_aspect('equal')
    ax_top.set_title(title, color='white', fontsize=7)
    ax_top.text(0.02, 0.97, 'top (XY)', transform=ax_top.transAxes,
                color='#aaaaaa', fontsize=6, va='top')

    # side view (XZ)
    ax_side.cla()
    style_ax(ax_side)
    ax_side.scatter(xyz[:, 0], xyz[:, 2], c=cols, s=1, linewidths=0)
    ax_side.set_aspect('equal')
    ax_side.text(0.02, 0.97, 'side (XZ)', transform=ax_side.transAxes,
                 color='#aaaaaa', fontsize=6, va='top')

    fig.canvas.draw_idle()


fig.canvas.mpl_connect('button_press_event', on_click)
plt.show()